# ✂️ ClipForge — free in your browser

Paste a YouTube link, get the most intense / funny / insightful moments as **vertical clips with animated captions**. Runs on Google's free GPU — nothing to install.

**How to use:** Runtime ▸ *Change runtime type* ▸ **T4 GPU** (optional but ~10× faster), then run the cells top to bottom (▶ on each, or Runtime ▸ Run all).

Source code & local install: https://github.com/WHERESHOULDISTART/clipforge

In [ ]:
#@title 1️⃣ Install ClipForge (≈1–2 min)
import os, subprocess, sys
if not os.path.exists('/content/clipforge'):
    !git clone -q https://github.com/WHERESHOULDISTART/clipforge /content/clipforge
%cd -q /content/clipforge
!git pull -q
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q . 2>&1 | grep -v -E 'WARNING|already satisfied' || true
# CUDA 12 runtime libs for CTranslate2 (Colab's system CUDA may be a different major version)
!pip install -q nvidia-cublas-cu12 'nvidia-cudnn-cu12>=9,<10' 2>&1 | grep -v -E 'WARNING|already satisfied' || true
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()
print('✅ Installed.', ('GPU: ' + gpu) if gpu else 'No GPU — transcription will be slower (Runtime ▸ Change runtime type ▸ T4 GPU).')

In [ ]:
#@title 2️⃣ Settings
#@markdown Paste a YouTube link with **people talking** (podcast, interview, vlog, stream, lecture). Only clip videos you have the rights to.
youtube_url = ""  #@param {type:"string"}
caption_style = "hormozi"  #@param ["hormozi", "mrbeast", "neon", "boxed", "minimal"]
number_of_clips = 3  #@param {type:"slider", min:1, max:10, step:1}
framing = "crop"  #@param ["crop", "blur", "fit"]
whisper_model = "small"  #@param ["tiny", "base", "small", "medium", "large-v3"]
#@markdown **Optional – smarter highlight picking & captions.** Leave blank to use the built-in heuristic (no key needed).
anthropic_api_key = ""  #@param {type:"string"}
openai_api_key = ""  #@param {type:"string"}

import os
os.environ['CLIPFORGE_WHISPER_MODEL'] = whisper_model
os.environ['CLIPFORGE_WORK_DIR'] = '/content/clipforge_output'
if anthropic_api_key: os.environ['ANTHROPIC_API_KEY'] = anthropic_api_key
if openai_api_key: os.environ['OPENAI_API_KEY'] = openai_api_key
os.environ['CLIPFORGE_LLM_PROVIDER'] = 'auto' if (anthropic_api_key or openai_api_key) else 'none'
print('✅ Settings saved. Highlight engine:', 'LLM' if os.environ['CLIPFORGE_LLM_PROVIDER']=='auto' else 'heuristic (free)')

In [ ]:
#@title 3️⃣ Make the clips 🎬
import os
from clipforge.pipeline import new_job, run_job

if not youtube_url.strip():
    raise SystemExit('👆 Paste a YouTube link into youtube_url in step 2️⃣, run that cell, then run this one again.')

job = new_job(youtube_url, style=caption_style, clip_count=number_of_clips, reframe=framing)
_last = {'s': None}
def _show(j):
    if j.status != _last['s'] or j.status == 'rendering':
        print(f"[{j.status:12}] {j.progress*100:5.1f}%  {j.message}")
        _last['s'] = j.status
run_job(job, _show)

if job.status == 'error':
    raise SystemExit(f'❌ {job.error}')
print(f'\n✅ {len(job.clips)} clips ready — job {job.id}')
for c in job.clips:
    h = c.highlight
    print(f"  {c.index}. [{h.start:6.1f}s → {h.end:6.1f}s]  🔥{h.score:3.0f}  {h.title}")

In [ ]:
#@title 4️⃣ Preview clips + captions
from IPython.display import HTML, display
import base64, html

for c in job.clips:
    data = base64.b64encode(open(c.video_path, 'rb').read()).decode()
    post = c.posts.get('tiktok')
    cap = html.escape(post.full_caption if post else '')
    display(HTML(f'''
    <div style="display:flex;gap:16px;align-items:flex-start;margin:12px 0;font-family:sans-serif">
      <video controls width="240" style="border-radius:12px;background:#000" src="data:video/mp4;base64,{data}"></video>
      <div><h3 style="margin:0 0 6px">{c.index}. {html.escape(c.highlight.title)}</h3>
      <p style="color:#888;margin:0 0 8px">{c.highlight.start:.0f}s → {c.highlight.end:.0f}s · {c.highlight.mood}</p>
      <pre style="white-space:pre-wrap;background:#1a1a1a;color:#eee;padding:10px;border-radius:8px;max-width:520px">{cap}</pre></div>
    </div>'''))

In [ ]:
#@title 5️⃣ Download everything (MP4s + SRTs + captions for every platform)
import json, shutil, os
from google.colab import files

out = f'/content/clipforge_output/clips/{job.id}'
with open(os.path.join(out, 'captions.md'), 'w') as f:
    for c in job.clips:
        f.write(f'# Clip {c.index}: {c.highlight.title}\n\n')
        for p, post in c.posts.items():
            f.write(f'## {p}\n**{post.title}**\n\n{post.full_caption}\n\n')
zip_path = shutil.make_archive(f'/content/clipforge_{job.id}', 'zip', out)
print('📦', zip_path)
files.download(zip_path)

## Posting

Colab has no desktop browser, so the one-click **Open in TikTok →** publisher isn't available here — download the zip, then upload the MP4s in each app and paste the matching caption from `captions.md`.

For the full experience (web dashboard + browser publishing to TikTok / Shorts / Reels / Facebook / X), install locally:

```bash
git clone https://github.com/WHERESHOULDISTART/clipforge && cd clipforge
pip install -e . && playwright install chromium
clipforge serve
```

⭐ Star the repo if this saved you time: https://github.com/WHERESHOULDISTART/clipforge